# 🧹 Module 3, 4 & 5: Data Preprocessing, Sparse Matrices, and Train-Test Splitting
**Movie Recommendation System Using Machine Learning**

---

### 🎯 Objectives
1. Clean data: Remove null values, duplicates, and validate rating ranges $[0.5, 5.0]$.
2. Filter movies with insufficient ratings to eliminate statistical noise.
3. Perform a reproducible **Train-Test Split** and learn how to **prevent data leakage**.
4. Construct a User-Item interaction matrix and convert it into a **SciPy CSR sparse matrix**.
5. Understand the memory and computational benefits of sparse representations.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

sys.path.insert(0, "..")
from src.preprocessing import (
    clean_movies,
    clean_ratings,
    filter_insufficient_ratings,
    train_test_split_ratings,
    build_user_item_matrix
)

print("Modules imported successfully!")


## 1. Cleaning Raw Data
We validate that:
- No IDs or ratings are null.
- Duplicate user ratings for the same movie are resolved.
- Ratings fall strictly within $[0.5, 5.0]$.


In [ ]:
raw_movies = pd.read_csv("../dataset/movies.csv")
raw_ratings = pd.read_csv("../dataset/ratings.csv")

cleaned_movies = clean_movies(raw_movies)
cleaned_ratings = clean_ratings(raw_ratings)

print(f"Cleaned Movies:  {len(cleaned_movies):,}")
print(f"Cleaned Ratings: {len(cleaned_ratings):,}")


## 2. Filtering Insufficient Ratings (Noise Reduction)
Movies with only 1 or 2 ratings provide almost zero co-occurrence overlap, creating noise.
We retain movies with at least 10 ratings.


In [ ]:
filtered_ratings, filtered_movies = filter_insufficient_ratings(
    cleaned_ratings,
    cleaned_movies,
    min_movie_ratings=10,
    min_user_ratings=5
)

print(f"Retained Movies:  {len(filtered_movies):,}")
print(f"Retained Ratings: {len(filtered_ratings):,}")
print(f"Retained Users:   {filtered_ratings['userId'].nunique():,}")


## 3. Train-Test Splitting & Preventing Data Leakage

### 🛡️ Why Data Leakage Prevention is Critical
In recommendation systems, data leakage occurs if the test ratings are visible when building
the interaction matrix or calculating similarity scores.
To prevent leakage:
1. Split ratings randomly (80% train, 20% test) using a fixed seed (`random_state=42`).
2. Train similarity matrices and popular lists **strictly on train ratings**.
3. Use test ratings **solely** to benchmark prediction accuracy.


In [ ]:
train_ratings, test_ratings = train_test_split_ratings(
    filtered_ratings, test_size=0.20, random_state=42
)

print(f"Training ratings: {len(train_ratings):,} ({len(train_ratings)/len(filtered_ratings)*100:.1f}%)")
print(f"Testing ratings:  {len(test_ratings):,} ({len(test_ratings)/len(filtered_ratings)*100:.1f}%)")

# Verify zero overlap
train_pairs = set(zip(train_ratings["userId"], train_ratings["movieId"]))
test_pairs = set(zip(test_ratings["userId"], test_ratings["movieId"]))
leakage = train_pairs.intersection(test_pairs)
print(f"Overlapping interaction pairs: {len(leakage)} (Zero indicates NO data leakage!)")


## 4. Building the User-Item Sparse Interaction Matrix

### Why SciPy CSR Matrix?
A dense matrix of 610 users $	imes$ 2,269 movies = 1.38 million cells, 98% of which are zeroes.
Compressed Sparse Row (`csr_matrix`) stores:
- Non-zero values (`data`)
- Column indices (`indices`)
- Row pointers (`indptr`)


In [ ]:
sparse_mat, u2idx, idx2u, m2idx, idx2m, pivot_df = build_user_item_matrix(
    train_ratings, filtered_movies
)

dense_size_bytes = sparse_mat.shape[0] * sparse_mat.shape[1] * 4  # float32 = 4 bytes
sparse_size_bytes = (sparse_mat.data.nbytes + sparse_mat.indices.nbytes + sparse_mat.indptr.nbytes)

print(f"Matrix Shape:               {sparse_mat.shape} (Users x Movies)")
print(f"Non-zero ratings stored:    {sparse_mat.nnz:,}")
print(f"Dense matrix memory:        {dense_size_bytes / 1024:.2f} KB")
print(f"Sparse CSR matrix memory:   {sparse_size_bytes / 1024:.2f} KB")
print(f"Memory reduction ratio:     {dense_size_bytes / sparse_size_bytes:.1f}x smaller!")


## 5. Beginner Exercises & Common Errors

### 📝 Exercises
1. Inspect the row in `sparse_mat` for user 1. How many movies did user 1 rate in the training set?
2. What happens if you try to index a movie that was filtered out? How does the mapping dictionary protect against index errors?

### ⚠️ Common Errors & Fixes
- **Error:** MemoryError when calling `sparse_matrix.toarray()`.
  - **Fix:** Never convert massive sparse matrices to dense arrays in memory; keep them as CSR/CSC matrices.
